# Spark Iceberg Notebook


In [1]:
import os

SPARK_CONNECT_HOST = os.environ["SPARK_CONNECT_HOST"]
SPARK_CONNECT_PORT = os.environ["SPARK_CONNECT_PORT"]

SPARK_CONNECT_SERVER = f"{SPARK_CONNECT_HOST}:{SPARK_CONNECT_PORT}"

print(f"SPARK_CONNECT_SERVER: {SPARK_CONNECT_SERVER}")

SPARK_CONNECT_SERVER: spark:15002


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("iceberg-usage") \
    .remote(f"sc://{SPARK_CONNECT_SERVER}") \
    .getOrCreate()
    
try:
    spark.sql("USE polaris")
    spark.sql("CREATE NAMESPACE IF NOT EXISTS COLLADO_TEST")
    spark.sql("CREATE NAMESPACE IF NOT EXISTS COLLADO_TEST.PUBLIC")
    spark.sql("SHOW NAMESPACES IN COLLADO_TEST").show()
    spark.sql("USE NAMESPACE COLLADO_TEST.PUBLIC")
    spark.sql("""CREATE TABLE IF NOT EXISTS TEST_TABLE (
        id bigint NOT NULL COMMENT 'unique id',
        data string)
    USING iceberg;
    """)
    spark.sql("SELECT * FROM TEST_TABLE").show()
    spark.sql("INSERT INTO TEST_TABLE VALUES (1, 'some data'), (2, 'more data'), (3, 'yet more data')")
    spark.sql("SELECT * FROM TEST_TABLE").show()
finally:
    spark.stop()

+-------------------+
|          namespace|
+-------------------+
|COLLADO_TEST.PUBLIC|
+-------------------+

+---+----+
| id|data|
+---+----+
+---+----+

+---+-------------+
| id|         data|
+---+-------------+
|  1|    some data|
|  2|    more data|
|  3|yet more data|
+---+-------------+

